# Anchors and key transitions

Checkpoints and key handovers on every wire major. Trusted key intervals and expected fingerprints are explicit; a changed chain hash is refused.

Open this file in JupyterLab or VS Code with the **Deno** kernel. From a repository checkout, install Deno 2.5.6, Python 3.13.9 and uv 0.9.9 (the `.tool-versions` pins), run `uv sync --frozen` and `uv run --frozen deno jupyter --install`, then select Deno and Run All. Any Jupyter-capable editor can use that installed kernel. See [launch and automated checks](README.md).

The saved notebook installs exact published packages. `pnpm notebooks:local` checks the packed candidate; `pnpm notebooks:published` runs these cells unchanged against npm. Fixed times describe a demonstration, not current credentials. Keys and signatures are generated, never golden output. Host authorization, replay and revocation stay outside this notebook.


In [1]:
import signerMetadata from "npm:@bounded-authority-protocol/signer@2.0.0/package.json" with { type: "json" };
import verifierMetadata from "npm:@bounded-authority-protocol/verifier@2.0.0/package.json" with { type: "json" };
if (signerMetadata.version !== "2.0.0" || verifierMetadata.version !== "2.0.0") throw new Error("package version mismatch");
console.log("exact signer and verifier releases loaded");

exact signer and verifier releases loaded


In [2]:
import { Buffer } from "node:buffer";
import { generateKeyPairSync, sign } from "node:crypto";
import { jwkFromPublicKey, thumbprint, v3 } from "npm:@bounded-authority-protocol/verifier@2.0.0";
import type { Es256KeyHandle, KeyHandle, KeyRole } from "npm:@bounded-authority-protocol/signer@2.0.0";

const fromBase64url = (text: string): Uint8Array => new Uint8Array(Buffer.from(text, "base64url"));

/** An Ed25519 handle over a fresh in-process key (contract-majors 1 and 2). */
function testKeyHandle(keyId: string, role: KeyRole = "holder"): KeyHandle {
  const { publicKey, privateKey } = generateKeyPairSync("ed25519");
  const raw = fromBase64url(publicKey.export({ format: "jwk" }).x!);
  return {
    sign: (message) => new Uint8Array(sign(null, message, privateKey)),
    publicKey: () => raw,
    thumbprint: () => thumbprint(jwkFromPublicKey(raw)),
    keyIdentity: () => ({ keyId, publicKey: raw }),
    signingIdentity: () => ({ role, keyId, publicKey: raw }),
  };
}

/** An ES256 handle over a fresh in-process P-256 key (contract-major 3). */
function testEs256KeyHandle(keyId: string, role: KeyRole = "holder"): Es256KeyHandle {
  const { publicKey, privateKey } = generateKeyPairSync("ec", { namedCurve: "P-256" });
  const { x, y } = publicKey.export({ format: "jwk" }) as { x: string; y: string };
  const raw = new Uint8Array([0x04, ...fromBase64url(x), ...fromBase64url(y)]);
  return {
    sign: (message) => new Uint8Array(sign("sha256", message, { key: privateKey, dsaEncoding: "ieee-p1363" })),
    publicKey: () => raw,
    thumbprint: () => v3.thumbprint({ crv: "P-256", kty: "EC", x, y }),
    keyIdentity: () => ({ keyId, publicKey: raw }),
    signingIdentity: () => ({ role, keyId, publicKey: raw }),
  };
}
console.log("demonstration key constructors ready");

demonstration key constructors ready


In [3]:
import { signAnchor, signKeyTransition } from "npm:@bounded-authority-protocol/signer@2.0.0";
import { publicKeyThumbprintRaw, verifyHistoricalAnchor, verifyKeyTransition } from "npm:@bounded-authority-protocol/verifier@2.0.0";

{


const current = testKeyHandle("audit-key-1");
const next = testKeyHandle("audit-key-2");
const chainHash = new Uint8Array(32); // the chain's hash at this sequence (all zero at genesis)

const anchor = await signAnchor(
  { anchorId: "urn:example:anchor:0", chainId: "urn:example:chain:audit-log", sequence: 0, chainHash },
  current, { anchoredAt: 1_767_225_600 },
);
if (!anchor.ok) throw new Error(`signAnchor: ${anchor.error}`);

// Rotate: the current key signs the handover to the next key.
const transition = await signKeyTransition({
  transitionId: "urn:example:transition:1", chainId: "urn:example:chain:audit-log",
  effectiveAt: 1_767_312_000, nextKeyId: "audit-key-2", nextPublicKey: await next.publicKey(),
}, current);
if (!transition.ok) throw new Error(`signKeyTransition: ${transition.error}`);

const currentKey = { keyId: "audit-key-1", publicKey: await current.publicKey(), validFrom: 0, validBefore: null };
const nextKey = { keyId: "audit-key-2", publicKey: await next.publicKey(), validFrom: 1_767_312_000, validBefore: null };
const anchorFacts = verifyHistoricalAnchor(anchor.value.anchor, currentKey, {
  anchorId: "urn:example:anchor:0", chainId: "urn:example:chain:audit-log", keyId: "audit-key-1",
  sequence: 0, anchoredAt: 1_767_225_600, chainHash, keyFingerprint: publicKeyThumbprintRaw(currentKey.publicKey),
});
const transitionFacts = verifyKeyTransition(transition.value.keyTransition, currentKey, nextKey, {
  transitionId: "urn:example:transition:1", chainId: "urn:example:chain:audit-log", effectiveAt: 1_767_312_000,
  currentKeyId: "audit-key-1", nextKeyId: "audit-key-2",
  currentKeyFingerprint: publicKeyThumbprintRaw(currentKey.publicKey),
  nextKeyFingerprint: publicKeyThumbprintRaw(nextKey.publicKey),
});
console.log(`anchor verifies: ${anchorFacts.ok}; transition verifies: ${transitionFacts.ok}`);
if (!anchorFacts.ok || !transitionFacts.ok) throw new Error("historical verification");
const wrongHash = verifyHistoricalAnchor(anchor.value.anchor, currentKey, { anchorId: "urn:example:anchor:0", chainId: "urn:example:chain:audit-log", keyId: "audit-key-1", sequence: 0, anchoredAt: 1_767_225_600, chainHash: new Uint8Array(32).fill(1), keyFingerprint: publicKeyThumbprintRaw(currentKey.publicKey) });
if (wrongHash.ok) throw new Error("changed checkpoint accepted");
}


anchor verifies: true; transition verifies: true


In [4]:
import { signV2Anchor, signV2KeyTransition } from "npm:@bounded-authority-protocol/signer@2.0.0";
import { publicKeyThumbprintRaw, v2 } from "npm:@bounded-authority-protocol/verifier@2.0.0";

{


const current = testKeyHandle("audit-key-1");
const next = testKeyHandle("audit-key-2");
const chainHash = new Uint8Array(32);

const anchor = await signV2Anchor(
  { anchorId: "urn:example:anchor:0", chainId: "urn:example:chain:audit-log", sequence: 0, chainHash },
  current, { anchoredAt: 1_767_225_600 },
);
const transition = await signV2KeyTransition({
  transitionId: "urn:example:transition:1", chainId: "urn:example:chain:audit-log",
  effectiveAt: 1_767_312_000, nextKeyId: "audit-key-2", nextPublicKey: await next.publicKey(),
}, current);
if (!anchor.ok || !transition.ok) throw new Error("signing failed");

const currentKey = { keyId: "audit-key-1", publicKey: await current.publicKey(), validFrom: 0, validBefore: null };
const nextKey = { keyId: "audit-key-2", publicKey: await next.publicKey(), validFrom: 1_767_312_000, validBefore: null };
const anchorFacts = v2.verifyHistoricalAnchor(anchor.value.anchor, currentKey, {
  anchorId: "urn:example:anchor:0", chainId: "urn:example:chain:audit-log", keyId: "audit-key-1",
  sequence: 0, anchoredAt: 1_767_225_600, chainHash, keyFingerprint: publicKeyThumbprintRaw(currentKey.publicKey),
});
const transitionFacts = v2.verifyKeyTransition(transition.value.keyTransition, currentKey, nextKey, {
  transitionId: "urn:example:transition:1", chainId: "urn:example:chain:audit-log", effectiveAt: 1_767_312_000,
  currentKeyId: "audit-key-1", nextKeyId: "audit-key-2",
  currentKeyFingerprint: publicKeyThumbprintRaw(currentKey.publicKey),
  nextKeyFingerprint: publicKeyThumbprintRaw(nextKey.publicKey),
});
console.log(`v2 anchor verifies: ${anchorFacts.ok}; v2 transition verifies: ${transitionFacts.ok}`);
if (!anchorFacts.ok || !transitionFacts.ok) throw new Error("v2 historical verification");
}


v2 anchor verifies: true; v2 transition verifies: true


In [5]:
import { signV3Anchor, signV3KeyTransition } from "npm:@bounded-authority-protocol/signer@2.0.0";
import { base64urlDecode, strUtf8, v3 } from "npm:@bounded-authority-protocol/verifier@2.0.0";

{


const current = testEs256KeyHandle("ledger-key-1");
const next = testEs256KeyHandle("ledger-key-2");
const fingerprint = async (handle: typeof current) => base64urlDecode(strUtf8(await handle.thumbprint()));
const chainHash = new Uint8Array(32); // the chain's hash at this sequence (all zero at genesis)

const anchor = await signV3Anchor(
  { anchorId: "urn:example:anchor:0", chainId: "urn:example:chain:ledger", sequence: 0, chainHash },
  current, { anchoredAt: 1_767_225_600 },
);
const transition = await signV3KeyTransition({
  transitionId: "urn:example:transition:1", chainId: "urn:example:chain:ledger",
  effectiveAt: 1_767_312_000, nextKeyId: "ledger-key-2", nextPublicKey: await next.publicKey(),
}, current);
if (!anchor.ok || !transition.ok) throw new Error("signing failed");

const currentKey = { keyId: "ledger-key-1", publicKey: await current.publicKey(), validFrom: 0, validBefore: null };
const nextKey = { keyId: "ledger-key-2", publicKey: await next.publicKey(), validFrom: 1_767_312_000, validBefore: null };
const anchorFacts = v3.verifyHistoricalAnchor(anchor.value.anchor, currentKey, {
  anchorId: "urn:example:anchor:0", chainId: "urn:example:chain:ledger", keyId: "ledger-key-1",
  sequence: 0, anchoredAt: 1_767_225_600, chainHash, keyFingerprint: await fingerprint(current),
});
const transitionFacts = v3.verifyKeyTransition(transition.value.keyTransition, currentKey, nextKey, {
  transitionId: "urn:example:transition:1", chainId: "urn:example:chain:ledger", effectiveAt: 1_767_312_000,
  currentKeyId: "ledger-key-1", nextKeyId: "ledger-key-2",
  currentKeyFingerprint: await fingerprint(current), nextKeyFingerprint: await fingerprint(next),
});
console.log(`v3 anchor verifies: ${anchorFacts.ok}; v3 transition verifies: ${transitionFacts.ok}`);
if (!anchorFacts.ok || !transitionFacts.ok) throw new Error("v3 historical verification");
}


v3 anchor verifies: true; v3 transition verifies: true


## Cleanup

These cells keep keys and compacts only in the kernel. Shut down the kernel to discard them. No private material is persisted. The runner shuts down the kernel and removes its consumer directory, package caches and temporary kernel registration, and fails on files left in the notebook working directory.

Continue with the [guides](../guides/getting-started.md).
